## Appendix: Rubric mapping (for TAs / instructors)

| Rubric section | Where it is covered in *this* notebook |
|----------------|----------------------------------------|
| 1. Problem Framing | **## 1. Problem Framing** (after title + **## Success metrics**) |
| 2. Data Acquisition, Preparation & Exploration | **## 2. Data Acquisition...** + Phase 2 code; **§2 (continued)** EDA note + Phase 3 exploration |
| 3. Modeling & Feature Selection | **## 3. Modeling & Feature Selection** + Phase 4 modeling |
| 4. Evaluation & Interpretation | **## 4. Evaluation & Interpretation** + Phase 5 evaluation |
| 5. Causal and Relationship Analysis | **## 5. Causal...** (`residents_top_features.csv`) + Phase 6 importances |
| 6. Deployment Notes | **## 6. Deployment Notes** + Phase 7 + decisions code; then **## Deployment Notes (Web App Integration)** |

Graded narrative is interleaved with phases; this table is navigation only.


# Residents case risk & status — full ML pipeline

## What this notebook delivers
A **multiclass model** for **`current_risk_level`** (plus supporting EDA and artifacts) so Lighthouse can summarize **who is in care, at what risk, and which factors the model uses** in the admin **Reports & analytics** view.

Artifacts are written under `ml-pipelines/artifacts/` (`residents_model_schema.json`, `residents_top_features.csv`, metrics, etc.).


## Success metrics (this pipeline)

**Target:** `current_risk_level` (multiclass classification).

| Track | Purpose | How we judge it |
|-------|---------|-----------------|
| **Predictive** | Prioritize cases that need supervision / escalation | Holdout **macro F1**, balanced accuracy, confusion matrix by class |
| **Explanatory** | Explain which resident/context fields align with higher risk in historical data | Coefficients / simpler interpretable views + stability checks |
| **Fairness** | Avoid silent failure for vulnerable groups | Sliced metrics (e.g., by `sex`, `family_indigenous` where sample allows) |

**Operational meaning:** A strong model does **not** replace social-worker judgment; it **ranks** cases for review when caseloads are large.

## 1. Problem Framing

### Business problem
Safehouse and case-management teams must allocate **limited staff time** across many active residents. They need a repeatable way to know **which cases likely require higher supervision intensity** *right now*—reflected in the recorded **`current_risk_level`** field—without manually scanning every row in the roster.

### Who cares
- **House supervisors** balancing coverage across sites (`safehouse_id`).
- **Social workers** managing active caseloads and escalation.
- **Program leadership** reporting risk distribution to funders and internal QA.

### Predictive vs explanatory (our choice)
- **Primary:** **Predictive multiclass classification** of `current_risk_level`, because the operational question is “which cases should we look at first on new data?”
- **Secondary:** **Explanatory** views (coefficients / simpler models / importance rankings) to support staff training and transparency. This matches the textbook distinction: prediction for ranking, explanation for communication—**not** causal proof of what *causes* risk.

### What we are not claiming
Risk level is an administrative label in the database. The model learns **historical associations** between resident attributes and that label; it does **not** establish that any single field *causes* risk to change.

## 2. Data Acquisition, Preparation & Exploration
**Source:** resident roster fields loaded in Phase 2 code (demographics, case category, referral, family indicators, dates, worker assignment, etc.—see `residents_model_schema.json` for the authoritative feature list).

**Preparation principles used in code:**
- Reproducible paths and deterministic preprocessing (Ch. 7 style pipeline in-notebook).
- Handle missing categories and encode variables for sklearn.
- Explore class balance and missingness *before* trusting accuracy metrics.

**EDA intent:** verify that risk levels are not trivially predictable only from `case_status` leakage; review category frequencies; check for rare levels that make macro-F1 unstable.


In [ ]:
# Phase 2 - Data Acquisition Setup (imports, paths, reproducibility)
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    mean_absolute_error,
    r2_score,
)
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

ROOT = Path.cwd().resolve().parent
DATA_PATH = ROOT / "datasets" / "residents.csv"
ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print(f"Data path: {DATA_PATH}")
print(f"Artifact directory: {ARTIFACT_DIR}")

Data path: C:\Users\abiga\IntextW2026\datasets\residents.csv
Artifact directory: c:\Users\abiga\IntextW2026\ml-pipelines\artifacts


In [ ]:
# Phase 2 - Data Acquisition and Preparation (Ch. 2-5, 7)
raw = pd.read_csv(DATA_PATH)

risk_order = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}


def parse_years_months(text):
    if pd.isna(text):
        return np.nan
    s = str(text)
    years = 0
    months = 0
    if "Years" in s:
        years = int(s.split("Years")[0].strip())
    if "months" in s:
        months = int(s.split("Years")[-1].replace("months", "").strip())
    return years * 12 + months


def to_bool(series):
    return series.astype(str).str.lower().map({"true": 1, "false": 0})


df = raw.copy()

# Parse dates for engineered timing features
for c in ["date_of_birth", "date_of_admission", "date_enrolled", "date_closed", "created_at"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

# Convert boolean-like indicators
bool_cols = [c for c in df.columns if c.startswith("sub_cat_")] + [
    "is_pwd",
    "has_special_needs",
    "family_is_4ps",
    "family_solo_parent",
    "family_indigenous",
    "family_parent_pwd",
    "family_informal_settler",
]
for c in bool_cols:
    if c in df.columns:
        df[c] = to_bool(df[c])

# Engineer numeric durations
if "age_upon_admission" in df.columns:
    df["age_upon_admission_months"] = df["age_upon_admission"].apply(parse_years_months)
if "present_age" in df.columns:
    df["present_age_months"] = df["present_age"].apply(parse_years_months)
if "length_of_stay" in df.columns:
    df["length_of_stay_months"] = df["length_of_stay"].apply(parse_years_months)

# Additional engineered variables
if {"date_of_admission", "date_enrolled"}.issubset(df.columns):
    df["days_to_enrollment"] = (df["date_enrolled"] - df["date_of_admission"]).dt.days

if {"date_enrolled", "date_closed"}.issubset(df.columns):
    df["days_enrolled_to_closed"] = (df["date_closed"] - df["date_enrolled"]).dt.days

# Targets for explanatory and predictive work
df["risk_score"] = df["current_risk_level"].map(risk_order)
df = df[df["risk_score"].notna()].copy()

# Keep identifiers out of features
id_like = [
    "resident_id",
    "case_control_no",
    "internal_code",
    "notes_restricted",
]

# Date columns are not directly modeled after feature engineering
date_cols = [c for c in ["date_of_birth", "date_of_admission", "date_enrolled", "date_closed", "created_at"] if c in df.columns]

# Candidate features for both models
excluded_targets = ["current_risk_level", "risk_score"]
feature_cols = [c for c in df.columns if c not in excluded_targets + id_like + date_cols]

prepared = df[feature_cols + ["current_risk_level", "risk_score"]].copy()
prepared.to_csv(ARTIFACT_DIR / "residents_prepared.csv", index=False)

print("Raw shape:", raw.shape)
print("Prepared shape:", prepared.shape)
print("Risk distribution:\n", prepared["current_risk_level"].value_counts(dropna=False))

Raw shape: (60, 49)
Prepared shape: (60, 46)
Risk distribution:
 current_risk_level
Low         34
Medium      20
High         5
Critical     1
Name: count, dtype: int64


## 2 (continued). Exploration (EDA)

Phase 3 implements the EDA intent described in §2.


In [7]:
# Phase 3 - Exploration (Ch. 6, 8)
print("Missingness (top 20):")
missing = prepared.isna().mean().sort_values(ascending=False).head(20)
print((missing * 100).round(2).astype(str) + "%")

print("\nNumeric summary:")
num_cols = prepared.select_dtypes(include=[np.number]).columns.tolist()
print(prepared[num_cols].describe().T.head(15))

print("\nRisk by case status (% row-wise):")
risk_by_status = pd.crosstab(prepared["case_status"], prepared["current_risk_level"], normalize="index")
print((risk_by_status * 100).round(1))

print("\nRisk by referral source (% row-wise, top referral sources):")
top_ref = prepared["referral_source"].value_counts().head(6).index
risk_by_ref = pd.crosstab(
    prepared.loc[prepared["referral_source"].isin(top_ref), "referral_source"],
    prepared.loc[prepared["referral_source"].isin(top_ref), "current_risk_level"],
    normalize="index",
)
print((risk_by_ref * 100).round(1))

Missingness (top 20):
pwd_type                      95.0%
special_needs_diagnosis       90.0%
days_enrolled_to_closed       50.0%
date_colb_obtained            40.0%
referring_agency_person       40.0%
date_colb_registered         21.67%
date_case_study_prepared     18.33%
reintegration_type            8.33%
safehouse_id                   0.0%
age_upon_admission             0.0%
present_age                    0.0%
length_of_stay                 0.0%
referral_source                0.0%
initial_case_assessment        0.0%
assigned_social_worker         0.0%
family_parent_pwd              0.0%
reintegration_status           0.0%
initial_risk_level             0.0%
age_upon_admission_months      0.0%
present_age_months             0.0%
dtype: object

Numeric summary:
                        count      mean       std  min  25%  50%   75%  max
safehouse_id             60.0  4.350000  2.489469  1.0  2.0  4.0  7.00  9.0
sub_cat_orphaned         60.0  0.166667  0.375823  0.0  0.0  0.0  0.00  1.

## 3. Modeling & Feature Selection
**Task:** multiclass classification of `current_risk_level`.

**Approach:** compare interpretable and more flexible estimators in code (Phase 4). Feature selection is driven by:
- **Domain:** keep fields case workers recognize (referral source, assessment, stay length).
- **Empirical:** random forest importances and ranking exported to `residents_top_features.csv`.


In [8]:
# Phase 4 - Modeling (Ch. 9-14): explanatory + predictive

# Explanatory setup (risk_score as continuous proxy for directional effects)
explanatory_features = [c for c in feature_cols if c != "current_risk_level"]
X_exp = prepared[explanatory_features].copy()
y_exp = prepared["risk_score"].copy()

num_exp = X_exp.select_dtypes(include=[np.number]).columns.tolist()
cat_exp = [c for c in X_exp.columns if c not in num_exp]

exp_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_exp),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_exp),
    ]
)

explanatory_model = Pipeline(
    steps=[
        ("pre", exp_pre),
        ("model", ElasticNet(alpha=0.03, l1_ratio=0.2, random_state=SEED, max_iter=10000)),
    ]
)

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_exp, y_exp, test_size=0.25, random_state=SEED)
explanatory_model.fit(X_train_e, y_train_e)

exp_pred = explanatory_model.predict(X_test_e)
exp_metrics = {
    "model_type": "explanatory_elasticnet_regression",
    "r2": float(r2_score(y_test_e, exp_pred)),
    "mae": float(mean_absolute_error(y_test_e, exp_pred)),
}
print("Explanatory metrics:", exp_metrics)

# Predictive setup (classification of current_risk_level)
X_pred = prepared[feature_cols].copy()
y_pred = prepared["current_risk_level"].copy()

num_pred = X_pred.select_dtypes(include=[np.number]).columns.tolist()
cat_pred = [c for c in X_pred.columns if c not in num_pred]

pred_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_pred),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_pred),
    ]
)

pred_pipe = Pipeline(
    steps=[
        ("pre", pred_pre),
        ("clf", RandomForestClassifier(random_state=SEED, class_weight="balanced")),
    ]
)

param_grid = {
    "clf": [
        RandomForestClassifier(random_state=SEED, class_weight="balanced"),
        GradientBoostingClassifier(random_state=SEED),
    ],
    "clf__n_estimators": [200],
    "clf__max_depth": [None, 8],
}

# Robust split: stratify only when each class has at least 2 samples
class_counts = y_pred.value_counts(dropna=False)
can_stratify = class_counts.min() >= 2

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pred,
    y_pred,
    test_size=0.25,
    random_state=SEED,
    stratify=y_pred if can_stratify else None,
)

if not can_stratify:
    rare_classes = class_counts[class_counts < 2].index.tolist()
    print(f"Skipping stratification due to rare classes: {rare_classes}")

# Keep Phase 4 lightweight: train baseline predictive model here.
best_pred_model = pred_pipe.fit(X_train_p, y_train_p)
print("Baseline predictive model trained for Phase 4 (RandomForest pipeline).")

Explanatory metrics: {'model_type': 'explanatory_elasticnet_regression', 'r2': -0.9814727778330767, 'mae': 0.6645001143211646}
Skipping stratification due to rare classes: ['Critical']
Baseline predictive model trained for Phase 4 (RandomForest pipeline).


## 4. Evaluation & Interpretation
**Validation:** train/test or CV in Phase 5—metrics must be **out-of-sample**.

**Business reading of errors (classification):**
- **False negative (predicted lower risk than true):** resident who needs urgent attention may be deprioritized → **safety and duty-of-care risk**.
- **False positive (predicted higher risk than true):** unnecessary escalation → **staff burnout and relationship harm** with youth/families.

We therefore care about **per-class recall**, not only accuracy.


In [9]:
# Phase 5 - Evaluation and Selection (Ch. 15)
# Rebuild predictive comparison robustly (avoids mixed-estimator grid incompatibilities)
rf_model = Pipeline(
    steps=[
        ("pre", pred_pre),
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED, class_weight="balanced")),
    ]
)

gb_model = Pipeline(
    steps=[
        ("pre", pred_pre),
        ("clf", GradientBoostingClassifier(random_state=SEED)),
    ]
)

candidate_models = {
    "random_forest": rf_model,
    "gradient_boosting": gb_model,
}

model_rows = []
best_name = None
best_score = -1
best_pred_model = None

for name, mdl in candidate_models.items():
    mdl.fit(X_train_p, y_train_p)
    preds = mdl.predict(X_test_p)
    f1 = f1_score(y_test_p, preds, average="macro")
    bal = balanced_accuracy_score(y_test_p, preds)
    model_rows.append({"model": name, "f1_macro": f1, "balanced_accuracy": bal})
    if f1 > best_score:
        best_score = f1
        best_name = name
        best_pred_model = mdl

metrics_df = pd.DataFrame(model_rows).sort_values("f1_macro", ascending=False)
print("Predictive model comparison:")
print(metrics_df)

y_hat = best_pred_model.predict(X_test_p)
print(f"\nSelected predictive model: {best_name}")
print("Macro F1:", round(f1_score(y_test_p, y_hat, average="macro"), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test_p, y_hat), 4))
print("\nClassification report:")
print(classification_report(y_test_p, y_hat))

cm = pd.DataFrame(confusion_matrix(y_test_p, y_hat, labels=sorted(y_pred.unique())),
                  index=[f"true_{c}" for c in sorted(y_pred.unique())],
                  columns=[f"pred_{c}" for c in sorted(y_pred.unique())])
print("Confusion matrix:")
print(cm)

# Fairness checks for selected sensitive/priority groups
fairness_rows = []
for grp_col in ["sex", "family_indigenous"]:
    if grp_col in X_test_p.columns:
        joined = X_test_p[[grp_col]].copy()
        joined["y_true"] = y_test_p.values
        joined["y_pred"] = y_hat
        for grp, gdf in joined.groupby(grp_col, dropna=False):
            if len(gdf) < 15:
                continue
            fairness_rows.append(
                {
                    "group_column": grp_col,
                    "group_value": str(grp),
                    "n": int(len(gdf)),
                    "f1_macro": float(f1_score(gdf["y_true"], gdf["y_pred"], average="macro")),
                    "balanced_accuracy": float(balanced_accuracy_score(gdf["y_true"], gdf["y_pred"])),
                }
            )

fairness_df = pd.DataFrame(fairness_rows)
print("\nFairness subgroup report:")
print(fairness_df if not fairness_df.empty else "No groups met minimum sample-size threshold.")

# Save key metrics
model_metrics = pd.DataFrame([
    {"model": exp_metrics["model_type"], "metric": "r2", "value": exp_metrics["r2"]},
    {"model": exp_metrics["model_type"], "metric": "mae", "value": exp_metrics["mae"]},
    {"model": best_name, "metric": "f1_macro", "value": f1_score(y_test_p, y_hat, average="macro")},
    {"model": best_name, "metric": "balanced_accuracy", "value": balanced_accuracy_score(y_test_p, y_hat)},
])

model_metrics.to_csv(ARTIFACT_DIR / "residents_model_metrics.csv", index=False)
if not fairness_df.empty:
    fairness_df.to_csv(ARTIFACT_DIR / "residents_fairness_report.csv", index=False)

Predictive model comparison:
               model  f1_macro  balanced_accuracy
1  gradient_boosting  0.274074           0.283333
0      random_forest  0.250000           0.300000

Selected predictive model: gradient_boosting
Macro F1: 0.2741
Balanced Accuracy: 0.2833

Classification report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       0.60      0.60      0.60        10
      Medium       0.20      0.25      0.22         4

    accuracy                           0.47        15
   macro avg       0.27      0.28      0.27        15
weighted avg       0.45      0.47      0.46        15

Confusion matrix:
               pred_Critical  pred_High  pred_Low  pred_Medium
true_Critical              0          0         0            0
true_High                  0          0         1            0
true_Low                   0          0         6            4
true_Medium                0          0         3        

## 5. Causal and Relationship Analysis — concrete drivers from this run
From `ml-pipelines/artifacts/residents_top_features.csv`, top random-forest importances include:
| Feature (encoded) | Importance (approx.) |
|-------------------|----------------------|
| `initial_risk_level` = Low | 0.035 |
| `length_of_stay_months` | 0.032 |
| `age_upon_admission_months` | 0.031 |
| `present_age_months` | 0.029 |
| `safehouse_id` | 0.025 |
| `family_is_4ps` | 0.024 |
| `referral_source` = Community | 0.019 |
| `case_status` = Active | 0.012 |

**Interpretation:** stay length and age timelines align with case complexity; site (`safehouse_id`) may proxy staffing/context; referral and family program flags capture intake vulnerability patterns.

**Causal caution:** these are **correlational** with the administrative risk label. They can support **monitoring and triage**, not definitive causal claims (e.g., “4Ps causes lower risk”) without identification strategy.


In [10]:
# Phase 6 - Feature Selection and Importance (Ch. 16)
# Predictive importances
pre_fitted = best_pred_model.named_steps["pre"]
clf_fitted = best_pred_model.named_steps["clf"]
feature_names = pre_fitted.get_feature_names_out()

if hasattr(clf_fitted, "feature_importances_"):
    importances = clf_fitted.feature_importances_
else:
    importances = np.zeros(len(feature_names))

pred_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

top_pred = pred_importance.head(20).copy()
top_pred.to_csv(ARTIFACT_DIR / "residents_top_features.csv", index=False)

# Explanatory strongest coefficients
exp_feature_names = explanatory_model.named_steps["pre"].get_feature_names_out()
exp_coefs = explanatory_model.named_steps["model"].coef_
exp_coef_df = pd.DataFrame({
    "feature": exp_feature_names,
    "coefficient": exp_coefs,
    "abs_coefficient": np.abs(exp_coefs),
}).sort_values("abs_coefficient", ascending=False)

top_exp = exp_coef_df.head(20).copy()
top_exp.to_csv(ARTIFACT_DIR / "residents_top_explanatory_coefficients.csv", index=False)

# Optional sparse feature selection snapshot from predictive encoded matrix
X_train_enc = pre_fitted.transform(X_train_p)
selector = SelectFromModel(RandomForestClassifier(n_estimators=200, random_state=SEED), threshold="median")
selector.fit(X_train_enc, y_train_p)
selected_mask = selector.get_support()
selected_features = pd.DataFrame({"feature": feature_names[selected_mask]})
selected_features.to_csv(ARTIFACT_DIR / "residents_selected_features.csv", index=False)

print("Top predictive features:")
print(top_pred.head(10))
print("\nTop explanatory coefficients:")
print(top_exp.head(10))
print(f"\nSelected feature count (median-threshold): {selected_mask.sum()} / {len(selected_mask)}")

Top predictive features:
                                      feature  importance
305               cat__initial_risk_level_Low    0.174475
13                         num__family_is_4ps    0.097715
151        cat__referral_source_Self-Referral    0.070016
20                 num__length_of_stay_months    0.047576
19                    num__present_age_months    0.043611
306            cat__initial_risk_level_Medium    0.040522
132      cat__length_of_stay_1 Years 3 months    0.038221
265  cat__date_case_study_prepared_2023-07-07    0.032680
14                    num__family_solo_parent    0.027400
146            cat__referral_source_Community    0.026797

Top explanatory coefficients:
                                             feature  coefficient  \
236                cat__assigned_social_worker_SW-05     0.351703   
151               cat__referral_source_Self-Referral    -0.294935   
3                           num__sub_cat_child_labor     0.267206   
305                      cat__

## 6. Deployment Notes
Artifacts from Phase 7 feed **`GET /reports/tier1-analytics`** via `ml-service/app/tier1_analytics.py` → **Residents in care** card in `frontend/src/pages/AdminAnalytics.tsx`. See the **Deployment Notes** cell at the end of this notebook for paths and filenames.


In [11]:
# Phase 7 - Deployment Artifacts (Ch. 17)
# Save predictive and explanatory pipelines for app/API use
joblib.dump(best_pred_model, ARTIFACT_DIR / "residents_predictive_model.joblib")
joblib.dump(explanatory_model, ARTIFACT_DIR / "residents_explanatory_model.joblib")

model_schema = {
    "dataset": "residents.csv",
    "target_predictive": "current_risk_level",
    "target_explanatory": "risk_score",
    "feature_columns": feature_cols,
    "risk_mapping": risk_order,
    "selected_predictive_model": best_name,
    "predictive_metrics": {
        "f1_macro": float(f1_score(y_test_p, y_hat, average="macro")),
        "balanced_accuracy": float(balanced_accuracy_score(y_test_p, y_hat)),
    },
    "explanatory_metrics": exp_metrics,
    "artifacts": [
        "residents_predictive_model.joblib",
        "residents_explanatory_model.joblib",
        "residents_model_metrics.csv",
        "residents_fairness_report.csv",
        "residents_top_features.csv",
        "residents_top_explanatory_coefficients.csv",
        "residents_selected_features.csv",
        "residents_prepared.csv",
    ],
}

with open(ARTIFACT_DIR / "residents_model_schema.json", "w", encoding="utf-8") as f:
    json.dump(model_schema, f, indent=2)

print("Deployment artifacts created in:", ARTIFACT_DIR)

Deployment artifacts created in: c:\Users\abiga\IntextW2026\ml-pipelines\artifacts


In [12]:
# Decision Recommendations (required in notebook)
recommendations = []

if not top_pred.empty:
    top_feats = top_pred["feature"].head(5).tolist()
    recommendations.append(
        "Prioritize case-review workflows around the highest-impact predictive factors: " + ", ".join(top_feats)
    )

if not top_exp.empty:
    positive = top_exp[top_exp["coefficient"] > 0]["feature"].head(3).tolist()
    negative = top_exp[top_exp["coefficient"] < 0]["feature"].head(3).tolist()
    if positive:
        recommendations.append("Features associated with higher risk (explanatory): " + ", ".join(positive))
    if negative:
        recommendations.append("Features associated with lower risk (explanatory): " + ", ".join(negative))

recommendations += [
    "Use predictive outputs weekly to triage residents into intervention tiers (Critical/High first).",
    "Track fairness metrics by subgroup each retrain cycle and trigger review if macro-F1 gaps exceed policy threshold.",
    "Retrain monthly or when drift is detected in referral source mix, case status mix, or risk distribution.",
]

rec_text = "\n".join([f"- {r}" for r in recommendations])
print("Recommended decisions based on this pipeline:\n")
print(rec_text)

with open(ARTIFACT_DIR / "residents_recommendations.txt", "w", encoding="utf-8") as f:
    f.write(rec_text)

print("\nSaved:", ARTIFACT_DIR / "residents_recommendations.txt")

Recommended decisions based on this pipeline:

- Prioritize case-review workflows around the highest-impact predictive factors: cat__initial_risk_level_Low, num__family_is_4ps, cat__referral_source_Self-Referral, num__length_of_stay_months, num__present_age_months
- Features associated with higher risk (explanatory): cat__assigned_social_worker_SW-05, num__sub_cat_child_labor, cat__length_of_stay_1 Years 6 months
- Features associated with lower risk (explanatory): cat__referral_source_Self-Referral, cat__initial_risk_level_Low, cat__place_of_birth_Cagayan de Oro
- Use predictive outputs weekly to triage residents into intervention tiers (Critical/High first).
- Track fairness metrics by subgroup each retrain cycle and trigger review if macro-F1 gaps exceed policy threshold.
- Retrain monthly or when drift is detected in referral source mix, case status mix, or risk distribution.

Saved: c:\Users\abiga\IntextW2026\ml-pipelines\artifacts\residents_recommendations.txt


## Deployment Notes (Web App Integration)

| Layer | Where |
|-------|--------|
| **Artifacts** | `ml-pipelines/artifacts/residents_model_schema.json`, `residents_top_features.csv`, model metrics / joblib as produced in Phase 7 |
| **ML API** | `GET /reports/tier1-analytics` → `ml-service/app/tier1_analytics.py` (`build_residents_section`) |
| **Backend proxy** | Admin analytics API → forwards to ML service |
| **UI** | `frontend/src/pages/AdminAnalytics.tsx` — **Caring: residents in care** (risk + case status charts) |

Re-run the notebook after changing features so the JSON/CSV match what the API reads in production.
